# Etapa 2: BiLSTM y XLM-R con el corpus bilingüe

Entrena con HumAID en inglés y HumAID traducido al español (`data/processed/bilingue/`, ver `scripts/prepare_humaid_es.py` y `scripts/prepare_bilingual.py`). Los conjuntos de evaluación y las métricas son los de la etapa 1, más las versiones traducidas de val, test y test_intra.

Contenido:

1. BiLSTM con la configuración de la etapa 1, reentrenado con el corpus bilingüe.
2. Ablaciones del BiLSTM: un cambio a la vez, decididas solo con val.
3. XLM-R con el corpus bilingüe.
4. Comparación con la etapa 1.

BERTweet no se reentrena: solo maneja inglés.

Antes de ejecutar en Kaggle: GPU e Internet activados, y `humaid_es.zip` (salida del notebook de traducción) agregado como Dataset en *Add Input*.

## 1. Repositorio, dependencias y datos

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_URL = "https://github.com/JuanTrujilloM/final-project-NN-DL.git"
BRANCH = "main"

# On Kaggle the repo is cloned; locally the notebook runs from the repo root.
if Path("/kaggle").exists():
    if not Path("final-project-NN-DL").exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("final-project-NN-DL")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if not Path("data/processed/humaid_en/train.csv").exists():
    subprocess.run([sys.executable, "scripts/prepare_data.py"], check=True)
print("Directorio de trabajo:", Path.cwd())

La traducción no está en el repositorio. En Kaggle se busca en los Datasets agregados como entrada y se copia a `data/raw/humaid_events_translated/`; después se pasa a `data/processed/humaid_es/` y se une con el inglés en `data/processed/bilingue/`.

In [ ]:
TRANSLATION_DIR = Path("data/raw/humaid_events_translated")
TRANSLATION_FILES = ["train.csv", "val.csv", "test.csv", "test_intra.csv"]


# Finds the folder with the four translated CSVs among the Kaggle inputs.
def find_translation(root=Path("/kaggle/input")):
    for path in sorted(root.rglob("test_intra.csv")):
        folder = path.parent
        if all((folder / name).exists() for name in TRANSLATION_FILES):
            if "text_es" in pd.read_csv(folder / "train.csv", nrows=1).columns:
                return folder
    return None


if not (TRANSLATION_DIR / "train.csv").exists() and Path("/kaggle/input").exists():
    source = find_translation()
    if source is not None:
        TRANSLATION_DIR.mkdir(parents=True, exist_ok=True)
        for name in TRANSLATION_FILES:
            shutil.copy(source / name, TRANSLATION_DIR / name)
        print("Traducción copiada desde", source)

if not (TRANSLATION_DIR / "train.csv").exists():
    raise SystemExit("No encuentro la traducción: agrega humaid_es.zip como Dataset de Kaggle o cópiala a "
                     f"{TRANSLATION_DIR}")
if not Path("data/processed/humaid_es/train.csv").exists():
    subprocess.run([sys.executable, "scripts/prepare_humaid_es.py"], check=True)
if not Path("data/processed/bilingue/train.csv").exists():
    subprocess.run([sys.executable, "scripts/prepare_bilingual.py"], check=True)

## 2. Librerías y configuración

In [ ]:
import gc
import json
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import sentencepiece as spm
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


def set_all_seeds(seed):
    os.environ["PL_GLOBAL_SEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)

In [ ]:
DATA_DIR = Path("data/processed")
BI_DIR = DATA_DIR / "bilingue"
RESULTS_DIR = Path("results/etapa2")
STAGE1_DIR = Path("results/etapa1")
MODELS_DIR = Path("models")
TEXT_COL = "text_clean"

# None to train with all data; a small number (e.g. 500) for a quick test.
LIMIT = None
# Skips models that already have results/etapa2/<model>/resumen.json (useful if the session drops).
SKIP_DONE = True

# What to run; each block can go in a different session.
RUN = {"bilstm": True, "xlmr": True}
# Name of the training set saved in each summary.
TRAIN_SET = "humaid_en+humaid_trad"

LABEL_NAMES = [
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "infrastructure_and_utility_damage",
    "injured_or_dead_people",
    "missing_or_found_people",
    "not_humanitarian",
    "other_relevant_information",
    "requests_or_urgent_needs",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
]
NUM_CLASSES = len(LABEL_NAMES)
LABEL2ID = {label: i for i, label in enumerate(LABEL_NAMES)}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Datos

- **Train**: `humaid_en` y `humaid_trad` (su traducción).
- **Val** (early stopping y selección de modelo): inglés y traducido juntos, para que el modelo no se ajuste a un solo idioma.
- **Evaluación**: los cuatro conjuntos de la etapa 1 con los mismos nombres (`val`, `test` y `test_intra` en inglés, y `chile_2014`), más `val_es`, `test_es` y `test_intra_es` traducidos.

In [ ]:
def read_csv(path):
    df = pd.read_csv(path, dtype={"tweet_id": str})
    df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
    df["label_id"] = df["label"].map(LABEL2ID).astype(int)
    return df


def limit_rows(df, by=None):
    if LIMIT is None:
        return df.reset_index(drop=True)
    if by is None:
        return df.sample(n=min(LIMIT, len(df)), random_state=SEED).reset_index(drop=True)
    return df.sample(frac=1, random_state=SEED).groupby(by).head(LIMIT).reset_index(drop=True)


train_df = limit_rows(read_csv(BI_DIR / "train.csv"), by="fuente")
bi_eval = {role: read_csv(BI_DIR / f"{role}.csv") for role in ["val", "test", "test_intra"]}

val_df = limit_rows(bi_eval["val"], by="lang")
eval_dfs = {}
for role, df in bi_eval.items():
    eval_dfs[role] = limit_rows(df[df["lang"] == "en"])
    eval_dfs[f"{role}_es"] = limit_rows(df[df["lang"] == "es"])
eval_dfs["chile_2014"] = limit_rows(read_csv(DATA_DIR / "chile2014_es" / "test.csv"))
eval_dfs = {name: eval_dfs[name] for name in ["val", "test", "test_intra", "chile_2014",
                                              "val_es", "test_es", "test_intra_es"]}

# The corpus script already checks this; it is repeated here so the notebook never trains with a leak.
eval_ids = set().union(*(set(df["tweet_id"]) for df in eval_dfs.values()))
assert not set(train_df["tweet_id"]) & eval_ids, "Hay tweets de evaluación en train"

print(train_df.groupby("fuente").size().to_string())
print(f"Val (en + es): {len(val_df):,}")
for name, df in eval_dfs.items():
    print(f"  {name}: {len(df):,}")

## 4. Métricas y reporte

In [ ]:
# F1-macro over the classes present in the true labels.
def compute_metrics(labels, preds):
    present = sorted(set(labels))
    return {
        "f1_macro": f1_score(labels, preds, labels=present, average="macro", zero_division=0),
        "accuracy": accuracy_score(labels, preds),
    }


# Saves metrics, per-class report and confusion matrix for one set.
def full_report(labels, preds, name, model_name, out_dir):
    present = sorted(set(labels))
    names = [LABEL_NAMES[i] for i in present]
    metrics = compute_metrics(labels, preds)
    metrics["n"] = len(labels)
    metrics["por_clase"] = classification_report(
        labels, preds, labels=present, target_names=names, output_dict=True, zero_division=0
    )
    (out_dir / f"metricas_{name}.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))

    print(f"\n{'=' * 60}\n  {model_name} | {name} — F1-macro: {metrics['f1_macro']:.4f} | accuracy: {metrics['accuracy']:.4f}\n{'=' * 60}")
    print(classification_report(labels, preds, labels=present, target_names=names, digits=4, zero_division=0))

    cm = confusion_matrix(labels, preds, labels=present)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    for ax, data, fmt, title in zip(
        axes, [cm, cm_norm], ["d", ".2f"], ["Conteos absolutos", "Proporción por clase (recall)"]
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
        ax.set_xlabel("Predicción")
        ax.set_ylabel("Real")
        ax.set_title(f"{model_name} | {name}\n{title}")
    plt.tight_layout()
    fig.savefig(out_dir / f"confusion_{name}.png", dpi=120)
    plt.show()
    plt.close(fig)
    return metrics

## 5. Callbacks

In [ ]:
# Saves the best model according to a monitored metric.
class ModelCheckpoint:
    def __init__(self, filepath, monitor="val_f1_macro", mode="max", verbose=True):
        self.filepath = filepath
        self.monitor = monitor
        self.verbose = verbose
        self.best = float("-inf") if mode == "max" else float("inf")
        self.mode = mode

    def step(self, metrics, model):
        value = metrics[self.monitor]
        improved = value > self.best if self.mode == "max" else value < self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  ✓ Mejor modelo guardado ({self.monitor}: {value:.4f})")
        return improved


# Stops training when the metric stops improving.
class EarlyStopping:
    def __init__(self, monitor="val_f1_macro", patience=5, min_delta=1e-4, mode="max"):
        self.monitor = monitor
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = float("inf") if mode == "min" else float("-inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == "min"
                    else value > self.best + self.min_delta)
        if improved:
            self.best = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  ⏹ EarlyStopping activado (sin mejora en {self.patience} épocas)")
        return improved


# Lowers the learning rate when the metric plateaus.
class ReduceLROnPlateau:
    def __init__(self, optimizer, monitor="val_f1_macro", patience=3, factor=0.5, mode="max"):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode=mode, patience=patience, factor=factor
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])

## 6. Utilidades de entrenamiento

In [ ]:
# Class weights from train: balanced, sqrt (softer) or none.
def class_weights(label_ids, mode="balanced"):
    if mode == "none":
        return None
    counts = np.bincount(np.asarray(label_ids), minlength=NUM_CLASSES).astype(float)
    weights = len(label_ids) / (NUM_CLASSES * np.maximum(counts, 1))
    if mode == "sqrt":
        weights = np.sqrt(weights)
    return torch.tensor(weights, dtype=torch.float).to(DEVICE)


# One epoch; trains if an optimizer is given, otherwise only evaluates.
def run_epoch(model, loader, criterion, forward_fn, optimizer=None, scheduler=None, scaler=None, clip_grad=1.0):
    training = optimizer is not None
    model.train() if training else model.eval()
    use_amp = scaler is not None
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(training):
        for batch in loader:
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits, labels = forward_fn(model, batch)
                loss = criterion(logits.float(), labels)

            if training:
                optimizer.zero_grad()
                if use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
                    optimizer.step()
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * labels.size(0)
            n += labels.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    metrics = compute_metrics(all_labels, all_preds)
    return total_loss / n, metrics["accuracy"], metrics["f1_macro"]


@torch.no_grad()
def evaluate_clf(model, loader, forward_fn):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        logits, labels = forward_fn(model, batch)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    return all_preds, all_labels


# F1-macro on val in English and in Spanish separately (only val, never test).
def val_by_language(model, make_loader, forward_fn):
    out = {}
    for lang in ["en", "es"]:
        part = val_df[val_df["lang"] == lang]
        preds, labels = evaluate_clf(model, make_loader(part), forward_fn)
        out[f"val_{lang}"] = compute_metrics(labels, preds)["f1_macro"]
    return out


def new_history():
    return {"epoch": [], "phase": [], "lr": [], "train_loss": [], "val_loss": [],
            "train_acc": [], "val_acc": [], "train_f1_macro": [], "val_f1_macro": []}


def log_epoch(history, epoch, phase, lr, train, val):
    for key, value in [("epoch", epoch), ("phase", phase), ("lr", lr),
                       ("train_loss", train[0]), ("train_acc", train[1]), ("train_f1_macro", train[2]),
                       ("val_loss", val[0]), ("val_acc", val[1]), ("val_f1_macro", val[2])]:
        history[key].append(value)
    print(f"Época {epoch:02d} [{phase}] — "
          f"train loss: {train[0]:.4f}, F1-macro: {train[2]:.4f} | "
          f"val loss: {val[0]:.4f}, F1-macro: {val[2]:.4f}, acc: {val[1]:.4f} | lr: {lr:.2e}")
    return {"val_loss": val[0], "val_acc": val[1], "val_f1_macro": val[2]}


def plot_history(history, model_name, out_dir):
    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)
    epochs = history["epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="val", linestyle="--")
    axes[1].plot(epochs, history["train_f1_macro"], label="train")
    axes[1].plot(epochs, history["val_f1_macro"], label="val", linestyle="--")
    axes[0].set_title(f"{model_name} — Loss")
    axes[1].set_title(f"{model_name} — F1-macro")
    for ax in axes:
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.set_xlabel("Época")
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_dir / "curvas.png", dpi=120)
    plt.show()
    plt.close(fig)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "entrenables": trainable}


def hardware_info(start_time):
    info = {
        "tiempo_entrenamiento_min": round((time.time() - start_time) / 60, 2),
        "plataforma": platform.platform(),
        "torch": torch.__version__,
        "cpu_nucleos": os.cpu_count(),
        "gpu": None,
    }
    if DEVICE.type == "cuda":
        props = torch.cuda.get_device_properties(0)
        info["gpu"] = props.name
        info["gpu_memoria_gb"] = round(props.total_memory / 1e9, 1)
        info["gpu_memoria_max_usada_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    return info


# Evaluates the best model on every eval set; predictions are saved without tweet text.
def final_evaluation(model, make_loader, forward_fn, model_name, out_dir):
    results = {}
    for name, df in eval_dfs.items():
        preds, labels = evaluate_clf(model, make_loader(df), forward_fn)
        metrics = full_report(labels, preds, name, model_name, out_dir)
        results[name] = {"f1_macro": metrics["f1_macro"], "accuracy": metrics["accuracy"], "n": metrics["n"]}
        pd.DataFrame({
            "tweet_id": df["tweet_id"].astype(str),
            "event": df["event"],
            "label": [LABEL_NAMES[i] for i in labels],
            "pred": [LABEL_NAMES[i] for i in preds],
        }).to_csv(out_dir / f"predicciones_{name}.csv", index=False)
    return results


def save_summary(out_dir, **summary):
    (out_dir / "resumen.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str))


def is_done(name):
    return SKIP_DONE and (RESULTS_DIR / name / "resumen.json").exists()


def free_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 7. BiLSTM bilingüe

La configuración base es la de la etapa 1 sin cambios (tokenizador por palabras, mismos hiperparámetros); solo cambian los datos. El tokenizador se ajusta siempre con train, nunca con val o test.

In [ ]:
BILSTM_BASE = {
    "name": "bilstm_bi_base",
    "tokenizer": "word",
    "vocab_size": 30000,
    "min_frequency": 2,
    "max_len": 80,
    "embedding_dim": 200,
    "hidden_size": 128,
    "num_layers": 2,
    "dropout": 0.3,
    "pooling": "attention",
    "batch_size": 64,
    "epochs": 30,
    "optimizer": "adam",
    "lr": 1e-3,
    "weight_decay": 0.0,
    "clip_grad": 1.0,
    "patience": 5,
    "lr_patience": 2,
    "lr_factor": 0.5,
    "class_weights": "balanced",
}

In [ ]:
PAD, UNK = "[PAD]", "[UNK]"


# Word-level (stage 1) or SentencePiece tokenizer, fitted on the training texts only.
def build_text_tokenizer(hp, texts, out_dir):
    texts = list(texts)
    if hp["tokenizer"] == "word":
        tok = Tokenizer(models.WordLevel(unk_token=UNK))
        tok.normalizer = normalizers.Lowercase()
        tok.pre_tokenizer = pre_tokenizers.Whitespace()
        tok.train_from_iterator(texts, trainers.WordLevelTrainer(
            vocab_size=hp["vocab_size"], min_frequency=hp["min_frequency"], special_tokens=[PAD, UNK]))
        tok.save(str(out_dir / "tokenizer.json"))
        return {"encode": lambda xs: [e.ids for e in tok.encode_batch(list(xs))],
                "vocab_size": tok.get_vocab_size(), "pad_id": tok.token_to_id(PAD), "unk_id": tok.token_to_id(UNK)}

    # Lowercased like the word tokenizer (nmt_nfkc_cf); the placeholders stay as single pieces.
    prefix = out_dir / "spm"
    spm.set_random_generator_seed(SEED)
    spm.SentencePieceTrainer.train(
        sentence_iterator=iter(texts), model_prefix=str(prefix), vocab_size=hp["vocab_size"],
        model_type="unigram", character_coverage=0.9995, normalization_rule_name="nmt_nfkc_cf",
        pad_id=0, unk_id=1, bos_id=-1, eos_id=-1, user_defined_symbols=["@user", "httpurl"],
        hard_vocab_limit=False, num_threads=os.cpu_count() or 1, minloglevel=2,
    )
    sp = spm.SentencePieceProcessor(model_file=f"{prefix}.model")
    return {"encode": lambda xs: sp.encode(list(xs)), "vocab_size": sp.get_piece_size(), "pad_id": 0, "unk_id": 1}


class TweetDataset(Dataset):
    def __init__(self, texts, labels, max_len, text_tok):
        self.ids = [ids[:max_len] or [text_tok["unk_id"]] for ids in text_tok["encode"](texts)]
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.ids[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


def lstm_loader_factory(hp, text_tok):
    def collate_fn(batch):
        sequences, labels = zip(*batch)
        lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
        padded = pad_sequence(sequences, batch_first=True, padding_value=text_tok["pad_id"])
        return padded, torch.stack(labels), lengths

    def make_loader(df, shuffle=False):
        dataset = TweetDataset(df[TEXT_COL], df["label_id"], hp["max_len"], text_tok)
        return DataLoader(dataset, batch_size=hp["batch_size"], shuffle=shuffle, collate_fn=collate_fn)
    return make_loader

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embedding_dim, hidden_size, num_layers, dropout, pad_idx,
                 pooling="attention"):
        super().__init__()
        self.pooling = pooling
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # Being bidirectional doubles the size of the LSTM output.
        if pooling == "attention":
            self.attention = nn.Linear(hidden_size * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, ids, lengths):
        emb = self.emb_dropout(self.embedding(ids))
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (h, _) = self.lstm(packed)

        if self.pooling == "last":
            rep = torch.cat([h[-2], h[-1]], dim=1)
        else:
            # Attention pooling, with padding masked out.
            out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=ids.size(1))
            mask = torch.arange(ids.size(1), device=ids.device)[None, :] < lengths[:, None]
            scores = self.attention(out).squeeze(-1).masked_fill(~mask, float("-inf"))
            weights = torch.softmax(scores, dim=1)
            rep = (weights.unsqueeze(-1) * out).sum(dim=1)

        return self.head(self.dropout(rep))


def forward_lstm(model, batch):
    ids, labels, lengths = batch
    return model(ids.to(DEVICE), lengths.to(DEVICE)), labels.to(DEVICE)

In [ ]:
# Trains one BiLSTM configuration and returns the best model with its val metrics (val only).
def train_bilstm(hp, run_name, out_root):
    set_all_seeds(SEED)
    out_dir = out_root / run_name
    ckpt_dir = MODELS_DIR / "etapa2" / "bilstm" / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    text_tok = build_text_tokenizer(hp, train_df[TEXT_COL], ckpt_dir)
    make_loader = lstm_loader_factory(hp, text_tok)
    train_loader, val_loader = make_loader(train_df, shuffle=True), make_loader(val_df)

    model = BiLSTMClassifier(
        vocab_size=text_tok["vocab_size"], num_classes=NUM_CLASSES, embedding_dim=hp["embedding_dim"],
        hidden_size=hp["hidden_size"], num_layers=hp["num_layers"], dropout=hp["dropout"],
        pad_idx=text_tok["pad_id"], pooling=hp["pooling"],
    ).to(DEVICE)
    print(f"\n[{run_name}] vocabulario: {text_tok['vocab_size']:,} | "
          f"parámetros entrenables: {count_parameters(model)['entrenables']:,}")

    criterion = nn.CrossEntropyLoss(weight=class_weights(train_df["label_id"], hp["class_weights"]))
    optim_cls = torch.optim.AdamW if hp["optimizer"] == "adamw" else torch.optim.Adam
    optimizer = optim_cls(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    checkpoint = ModelCheckpoint(str(ckpt_dir / "model.pt"))
    reduce_lr = ReduceLROnPlateau(optimizer, patience=hp["lr_patience"], factor=hp["lr_factor"])
    early_stop = EarlyStopping(patience=hp["patience"])

    history = new_history()
    start_time = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(1, hp["epochs"] + 1):
        train_m = run_epoch(model, train_loader, criterion, forward_lstm, optimizer, clip_grad=hp["clip_grad"])
        val_m = run_epoch(model, val_loader, criterion, forward_lstm)
        metrics = log_epoch(history, epoch, "bilstm", optimizer.param_groups[0]["lr"], train_m, val_m)
        checkpoint.step(metrics, model)
        reduce_lr.step(metrics)
        early_stop.step(metrics)
        if early_stop.should_stop:
            break

    resources = hardware_info(start_time)
    plot_history(history, run_name, out_dir)
    model.load_state_dict(torch.load(ckpt_dir / "model.pt", map_location=DEVICE))
    return {
        "model": model, "make_loader": make_loader, "hp": hp, "recursos": resources,
        "vocabulario": text_tok["vocab_size"], "epocas": len(history["epoch"]),
        "val_f1_macro": checkpoint.best, **val_by_language(model, make_loader, forward_lstm),
    }


# Final evaluation on every set, written to results/etapa2/<name>/ like stage 1.
def report_bilstm(run, name, extra=None):
    out_dir = RESULTS_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    for path in (RESULTS_DIR / "ablaciones_bilstm" / run["run_name"]).glob("*"):
        shutil.copy(path, out_dir / path.name)
    results = final_evaluation(run["model"], run["make_loader"], forward_lstm, name, out_dir)
    save_summary(
        out_dir,
        nombre=name,
        conjunto_train=TRAIN_SET,
        mejor_val_f1_macro=run["val_f1_macro"],
        epocas_entrenadas=run["epocas"],
        parametros=count_parameters(run["model"]),
        vocabulario=run["vocabulario"],
        recursos=run["recursos"],
        resultados=results,
        hiperparametros={**run["hp"], "name": name},
        **(extra or {}),
    )
    return results

## 8. Ablaciones del BiLSTM

Se parte de la configuración base y se prueba **un cambio a la vez**. Un cambio se conserva solo si mejora el F1-macro de val (inglés + español) en al menos `MIN_GAIN`; el siguiente cambio se prueba sobre la mejor configuración hasta ese momento. Las decisiones se toman solo con val: test y Chile 2014 se usan una vez, al final, con el modelo base y con el mejor.

| Paso | Cambio | Motivo |
|---|---|---|
| 1 | SentencePiece (16k, unigram) y `max_len` 128 | Con subpalabras, una palabra en español sin ver no queda como `[UNK]`. El `max_len` sube porque un tweet tiene más subpalabras que palabras |
| 2 | AdamW con `weight_decay` 0,01 | Regularización |
| 3 | Dropout 0,5 | Regularización |
| 4 | Pesos de clase `sqrt` | Pesos menos extremos para las clases pequeñas |
| 5 | `hidden_size` 256 | Capacidad |

In [ ]:
ABLATIONS = [
    ("sentencepiece", {"tokenizer": "sentencepiece", "vocab_size": 16000, "max_len": 128}),
    ("adamw_wd", {"optimizer": "adamw", "weight_decay": 0.01}),
    ("dropout_0.5", {"dropout": 0.5}),
    ("pesos_sqrt", {"class_weights": "sqrt"}),
    ("hidden_256", {"hidden_size": 256}),
]
MIN_GAIN = 0.002

In [ ]:
if RUN["bilstm"] and not (is_done("bilstm_bi_base") and is_done("bilstm_bi")):
    ablation_root = RESULTS_DIR / "ablaciones_bilstm"
    rows = []

    def ablation_row(step, change, run, kept):
        row = {"paso": step, "cambio": change, "val_f1_macro": round(run["val_f1_macro"], 4),
               "val_en": round(run["val_en"], 4), "val_es": round(run["val_es"], 4),
               "epocas": run["epocas"], "minutos": run["recursos"]["tiempo_entrenamiento_min"],
               "gpu": run["recursos"]["gpu"], "se_conserva": kept}
        print(f"  → {change}: val F1 {row['val_f1_macro']:.4f} (en {row['val_en']:.4f}, es {row['val_es']:.4f})"
              f" — {'se conserva' if kept else 'se descarta'}")
        return row

    base = train_bilstm(BILSTM_BASE, "a0_base", ablation_root)
    base["run_name"] = "a0_base"
    rows.append(ablation_row(0, "base (configuración de la etapa 1)", base, True))
    best = base

    for step, (change, params) in enumerate(ABLATIONS, start=1):
        run_name = f"a{step}_{change}"
        run = train_bilstm({**best["hp"], **params}, run_name, ablation_root)
        run["run_name"] = run_name
        kept = run["val_f1_macro"] >= best["val_f1_macro"] + MIN_GAIN
        rows.append(ablation_row(step, change, run, kept))
        if kept:
            if best is not base:
                del best["model"]
            best = run
        else:
            del run["model"]
        free_memory()

    ablations = pd.DataFrame(rows)
    ablations.to_csv(RESULTS_DIR / "ablaciones_bilstm.csv", index=False)
    display(ablations)

    results_bilstm_base = report_bilstm(base, "bilstm_bi_base")
    kept_changes = [r["cambio"] for r in rows[1:] if r["se_conserva"]]
    results_bilstm = report_bilstm(best, "bilstm_bi", extra={"cambios_conservados": kept_changes})
    print("Cambios conservados:", kept_changes or "ninguno")
    del base, best
    free_memory()

## 9. XLM-R bilingüe

Mismos hiperparámetros que en la etapa 1 (fine-tuning en dos fases); solo cambian los datos de entrenamiento.

In [ ]:
class PretrainedTweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=max_len)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: values[idx] for key, values in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


class PretrainedClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1, freeze_encoder=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        # BERT uses token_type_ids; RoBERTa, BERTweet and XLM-R do not.
        cfg = self.encoder.config
        self.use_token_type_ids = getattr(cfg, "type_vocab_size", 0) > 1

        if freeze_encoder:
            self.freeze_encoder()

        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(cfg.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if self.use_token_type_ids and token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        out = self.encoder(**kwargs)
        if getattr(out, "pooler_output", None) is not None:
            cls_repr = out.pooler_output
        else:
            cls_repr = out.last_hidden_state[:, 0, :]
        return self.classifier(self.drop(cls_repr))

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True


def forward_transformer(model, batch):
    ttype = batch.get("token_type_ids")
    logits = model(
        batch["input_ids"].to(DEVICE),
        batch["attention_mask"].to(DEVICE),
        ttype.to(DEVICE) if ttype is not None else None,
    )
    return logits, batch["labels"].to(DEVICE)

In [ ]:
# Two-phase fine-tuning: frozen encoder first, then the whole model.
def finetune(hp, train_df):
    set_all_seeds(SEED)
    out_dir = RESULTS_DIR / hp["name"]
    ckpt_dir = MODELS_DIR / "etapa2" / hp["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    use_amp = hp["amp"] and DEVICE.type == "cuda"

    tokenizer = AutoTokenizer.from_pretrained(hp["model_name"], **hp.get("tokenizer_kwargs", {}))
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    num_workers = 2 if DEVICE.type == "cuda" else 0

    def make_loader(df, shuffle=False):
        dataset = PretrainedTweetDataset(df[TEXT_COL], df["label_id"], tokenizer, hp["max_len"])
        return DataLoader(dataset, batch_size=hp["batch_size"], shuffle=shuffle,
                          collate_fn=collator, num_workers=num_workers)

    train_loader = make_loader(train_df, shuffle=True)
    val_loader = make_loader(val_df)

    model = PretrainedClassifier(hp["model_name"], NUM_CLASSES, dropout=hp["dropout"],
                                 freeze_encoder=hp["epochs_frozen"] > 0).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_df["label_id"], hp["class_weights"]))
    scaler = torch.amp.GradScaler("cuda") if use_amp else None
    checkpoint = ModelCheckpoint(str(ckpt_dir / "model.pt"))
    history = new_history()
    epoch = 0
    start_time = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    # Phase 1: frozen encoder, only the classifier is trained.
    if hp["epochs_frozen"] > 0:
        print(f"\n[Fase 1 — encoder congelado] {count_parameters(model)['entrenables']:,} parámetros entrenables")
        opt_head = torch.optim.Adam(model.classifier.parameters(), lr=hp["lr_head"])
        for _ in range(hp["epochs_frozen"]):
            epoch += 1
            train_m = run_epoch(model, train_loader, criterion, forward_transformer, opt_head,
                                scaler=scaler, clip_grad=hp["clip_grad"])
            val_m = run_epoch(model, val_loader, criterion, forward_transformer, scaler=scaler)
            metrics = log_epoch(history, epoch, "congelado", hp["lr_head"], train_m, val_m)
            checkpoint.step(metrics, model)
        model.load_state_dict(torch.load(ckpt_dir / "model.pt", map_location=DEVICE))

    # Phase 2: whole model with a small lr and linear warmup.
    model.unfreeze_encoder()
    print(f"\n[Fase 2 — fine-tuning completo] {count_parameters(model)['entrenables']:,} parámetros entrenables")
    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": hp["lr_encoder"]},
        {"params": model.classifier.parameters(), "lr": hp["lr_encoder"] * 5},
    ], weight_decay=hp["weight_decay"])
    total_steps = len(train_loader) * hp["epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * hp["warmup_ratio"]), num_training_steps=total_steps
    )
    early_stop = EarlyStopping(patience=hp["patience"])
    for _ in range(hp["epochs"]):
        epoch += 1
        train_m = run_epoch(model, train_loader, criterion, forward_transformer, optimizer,
                            scheduler=scheduler, scaler=scaler, clip_grad=hp["clip_grad"])
        val_m = run_epoch(model, val_loader, criterion, forward_transformer, scaler=scaler)
        metrics = log_epoch(history, epoch, "fine-tuning", optimizer.param_groups[0]["lr"], train_m, val_m)
        checkpoint.step(metrics, model)
        early_stop.step(metrics)
        if early_stop.should_stop:
            break

    resources = hardware_info(start_time)
    plot_history(history, hp["name"], out_dir)

    model.load_state_dict(torch.load(ckpt_dir / "model.pt", map_location=DEVICE))
    results = final_evaluation(model, make_loader, forward_transformer, hp["name"], out_dir)
    save_summary(
        out_dir,
        nombre=hp["name"],
        modelo=hp["model_name"],
        conjunto_train=TRAIN_SET,
        tweets_train=train_df["fuente"].value_counts().to_dict(),
        mejor_val_f1_macro=checkpoint.best,
        val_por_idioma=val_by_language(model, make_loader, forward_transformer),
        epocas_entrenadas=len(history["epoch"]),
        parametros=count_parameters(model),
        recursos=resources,
        resultados=results,
        hiperparametros=hp,
    )
    print(f"Recursos: {resources}")

    # Free GPU memory before the next model.
    del model, optimizer
    free_memory()
    return results

In [ ]:
# Same values as stage 1 (BERTWEET dict with xlm-roberta-base).
XLMR = {
    "name": "xlmr_bi",
    "model_name": "xlm-roberta-base",
    "tokenizer_kwargs": {},
    "max_len": 128,
    "dropout": 0.1,
    "batch_size": 32,
    "epochs_frozen": 1,
    "lr_head": 1e-3,
    "epochs": 4,
    "lr_encoder": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "clip_grad": 1.0,
    "patience": 2,
    "class_weights": "balanced",
    "amp": True,
}

In [ ]:
if RUN["xlmr"] and not is_done(XLMR["name"]):
    finetune(XLMR, train_df)
elif RUN["xlmr"]:
    print(f"{XLMR['name']}: ya tiene resultados, se omite")

## 10. Comparación con la etapa 1

Las columnas `F1 val`, `F1 test`, `F1 test_intra` y `F1 chile_2014` son los mismos conjuntos de la etapa 1, así que se comparan directo. Las columnas `_es` solo existen en la etapa 2.

In [ ]:
rows = []
for path in sorted(RESULTS_DIR.glob("*/resumen.json")):
    summary = json.loads(path.read_text())
    row = {
        "etapa": 2,
        "modelo": summary["nombre"],
        "train": summary.get("conjunto_train"),
        "épocas": summary["epocas_entrenadas"],
        "minutos": summary["recursos"]["tiempo_entrenamiento_min"],
        "gpu": summary["recursos"]["gpu"],
    }
    row.update({f"F1 {name}": round(r["f1_macro"], 4) for name, r in summary["resultados"].items()})
    rows.append(row)
stage2 = pd.DataFrame(rows)
stage2.to_csv(RESULTS_DIR / "comparacion.csv", index=False)

stage1 = pd.read_csv(STAGE1_DIR / "comparacion.csv").assign(etapa=1, train="humaid_en")
comparison = pd.concat([stage1, stage2], ignore_index=True)
cols = ["etapa", "modelo", "train", "épocas", "minutos", "gpu", "F1 val", "F1 test", "F1 test_intra",
        "F1 chile_2014", "F1 val_es", "F1 test_es", "F1 test_intra_es"]
comparison = comparison[[c for c in cols if c in comparison]]
comparison.to_csv(RESULTS_DIR / "comparacion_etapas.csv", index=False)
comparison

In [ ]:
# Change from stage 1 to stage 2 for each model family (positive = better in stage 2).
pairs = [("bilstm_en", "bilstm_bi_base"), ("bilstm_en", "bilstm_bi"), ("xlmr_en", "xlmr_bi")]
metrics = ["F1 val", "F1 test", "F1 test_intra", "F1 chile_2014"]
table = comparison.set_index("modelo")
deltas = pd.DataFrame([
    {"etapa 1": a, "etapa 2": b, **{m: round(table.loc[b, m] - table.loc[a, m], 4) for m in metrics}}
    for a, b in pairs if a in table.index and b in table.index
])
deltas

In [ ]:
if Path("/kaggle").exists():
    shutil.make_archive("/kaggle/working/resultados_etapa2", "zip", RESULTS_DIR)
    print("Resultados comprimidos en /kaggle/working/resultados_etapa2.zip")